In [ ]:
#查看 python 版本
import sys
print(sys.version)

# 安装依赖库

In [ ]:
!pip install gym gym_super_mario_bros opencv-python==4.7.0.72 spinup joblib pyvirtualdisplay swig
!sudo apt-get install xvfb python3.11-dev swig -y
!sudo apt-get -y install python3-opengl -y
!sudo apt-get install xserver-xephyr -y #Xephyr 是一个嵌入式 X 服务器，它可以用于创建虚拟显示环境

#安装的 spinup ，会导致 Supermariobros-PPO-pytorch 的 test_lstm.py 不能运行，提示：cannot import name 'EpochLogger' from 'spinup'

#或者自己编译 openai 的 spinup 库，安装 spinup 库，按照如下步骤
# OpenMPI 安装
!sudo apt-get update && sudo apt-get install libopenmpi-dev -y

#安装 Spinning Up 

#参考链接：  https://spinningup.openai.com/en/latest/user/installation.html
!git clone https://github.com/openai/spinningup.git


#以上调整能正常安装，但 gpu 不能使用，再尝试如下命令，重新安装 torch
!pip install torch==2.2.0+cu121 torchvision==0.17.0+cu121 torchaudio==2.2.0 --extra-index-url https://download.pytorch.org/whl/cu121


#由于时间所用的库比较老，第一次安装会失败，调整了一下 setup.py，然后再次尝试安装：
#'cloudpickle==2.2.0',
#'gym[atari,box2d,classic_control]~=0.20.0',  
#'numpy==1.26.0',
#'tensorflow==2.18.0',  
#'torch==2.2.0+cu121',

#调整文件 spinningup\spinup\utils\mpi_tf.py,用 tf.keras.optimizers.Adam 替换 tf.train.AdamOptimizer, 因为再 tensorflow 2 版本以上做了调整

#问题说明：
#如果安装 gym==0.20.0 出现问题，可以尝试如下命令：
#pip install gym==0.20.0 -i https://pypi.org/simple
#如果还安装不上，使用 venv   : 
#python -m venv myenv
#source myenv/bin/activate



In [ ]:
#完成以上操作和调整，然后执行 spinningup 安装
!cd spinningup
!pip install -e .

#super_mario related package:
!pip install gym_super_mario_bros
!pip install opencv-python==4.7.0.72

如果执行训练时候，出现如下错误：
Traceback (most recent call last):
  File "/mnt/workspace/Supermariobros-PPO-pytorch/ppo_lstm.py", line 447, in <module>
    ppo(env_fn, actor=userActor, critic=userCritic,#core.MLPActorCritic, #gym.make(args.env)
  File "/mnt/workspace/Supermariobros-PPO-pytorch/ppo_lstm.py", line 257, in ppo
    ac_pi.to(device)
  File "/mnt/workspace/spinningup/myenv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1152, in to
    return self._apply(convert)
  File "/mnt/workspace/spinningup/myenv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 802, in _apply
    module._apply(fn)
  File "/mnt/workspace/spinningup/myenv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 802, in _apply
    module._apply(fn)
  File "/mnt/workspace/spinningup/myenv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 825, in _apply
    param_applied = fn(param)
  File "/mnt/workspace/spinningup/myenv/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1150, in convert
    return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)
RuntimeError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.

出现这个问题，有可能是 tensorflow 和 pytorch 冲突
解决方法：删除 spinningup 和 Supermariobros-PPO-pytorch ,重新配置环境。如果使用了 python venv, 也删除。
其中，tensorflow 版本2.0 以下找不到，调整为如下：
'tensorflow>=1.8.0,<2.20',
        'torch==2.2.0',
spinup/utils/mpi_tf.py 将 tf.train.AdamOptimizer 调整为 tf.keras.optimizers.Adam

# 随机动作播放超级玛丽

In [ ]:
from nes_py.wrappers import JoypadSpace
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

#使用gym_super_mario_bros包函数创建游戏环境env
env = gym_super_mario_bros.make('SuperMarioBros-v0')

#指定环境为简单模式（动作简化，去除一些左上、左下等复杂动作）
env = JoypadSpace(env, SIMPLE_MOVEMENT)

#使用gym的wrapper函数对游戏视频进行录像（由于notebook不支持display，我们录像后播放观看）
from gym import wrappers
env = wrappers.Monitor(env,"./gym-results", force=True)

#执行5000个简单的向右随机操作 
done = True #游戏结束标志
for step in range(5000):
    if done:
        #如果游戏结束则重置：
        state = env.reset() 
    state, reward, done, info = env.step(env.action_space.sample())

env.close()
print("Done, execute below code to view video")

**播放视频,可在 Jupeter notebook 中播放**

In [ ]:

import io
import base64
from IPython.display import HTML

video = io.open('./gym-results/openaigym.video.%s.video000000.mp4'%env.file_infix, 'r+b').read()
encoded = base64.b64encode(video)
HTML(data='''
    <video width="360" height="auto" alt="test" controls><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>'''
.format(encoded.decode('ascii')))

**R2 完整代码通关play超级玛丽**

In [ ]:
#下载我提前训练好的权重和代码
!git clone https://github.com/gaoxiaos/Supermariobros-PPO-pytorch.git

In [ ]:
#运行play测试程序, test.py 不存在，下列代码不通过
!cd Supermariobros-PPO-pytorch
!python test.py

In [ ]:
#查看运行录像
video = io.open('./gym-results/openaigym.video.%s.video000000.mp4' % env.file_infix, 'r+b').read()
encoded = base64.b64encode(video)
HTML(data='''
    <video width="360" height="auto" alt="test" controls><source src="data:video/mp4;base64,{0}" type="video/mp4" /></video>'''
.format(encoded.decode('ascii')))

In [ ]:
#创建倒立摆'CartPole-v0' env
import gym
env=gym.make('CartPole-v0')

#初始化游戏环境
env.reset()

#从上面的返回可以看到我们在执行环境初始化\重置时env返回给我们了初始化后的环境状态为：
#[ 0.03749292, -0.03226631, 0.01609263, -0.04661368] 这四个数字组成的状态变量（state variables）分别含义如下：
#0.03749292： 小车在轨道上的位置（position of the cart on the track） 
#-0.03226631： 杆子与竖直方向的夹角（angle of the pole with the vertical） 
#0.01609263： 小车速度（cart velocity） 
#-0.04661368： 角度变化率（rate of change of the angle）

In [ ]:
#环境包含的动作有哪些？
print("env.action_space: ", env.action_space)

从结果来看动作空间为2，也就是说倒立摆这个环境只有两个动作可以操作，分别是0和1 （向左和向右）从倒立摆的动画不难理解，通过左右移动来保持倒立摆不倒

In [ ]:
#执行一个向左的操作
obj, reward, done, info = env.step(0) #1 向右 0向左
print("obj", obj)
print("reward", reward)
print("done", done)
print("info", info)

#一个动作执行后，环境会返回四个变量
#（obj:新的状态（对照前面环境初始化的状态理解）、
#reward：指定该动作获得的奖励值（在游戏中的得分）、
#done:回合是否结束（你控制的小人是不是死了，对应回合结束）、
#info:额外信息（该游戏较简单，info为空））

In [ ]:
#随机获取一个动作
action = env.action_space.sample()
print(action)
#通过sample（）函数可以快速得到一个随机动作，由于该游戏动作空间为2，所以sample得到的值为0或1

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

#这里是创建一个 Display 对象，即虚拟显示器
#visible=0: 这个参数指定是否在屏幕上实际显示虚拟显示器的窗口。值为 0 表示该虚拟显示器是不可见的，适合于后台任务或无需人工干预的情况。
#size=(1400, 900): 设置虚拟显示器的分辨率大小，这里的分辨率为宽度 1400 像素，高度 900 像素
virtual_display = Display(visible=1, size=(1400, 900))
virtual_display.start()

In [ ]:
#创建倒立摆'CartPole-v0' env
import gym
env = gym.make('CartPole-v0')

#运行1000组随机动作
from gym import wrappers
env = wrappers.Monitor(env,"./gym-results", force=True)
#初始化游戏环境
env.reset()
for _ in range(1000):
    env.render() #服务器上无display,不支持render
    obj, reward, done, info = env.step(env.action_space.sample()) # take a random action
    if done:
        env.reset()
env.close()

# 超级玛丽环境讲解
超级玛丽主要区别于倒立摆游戏的是超级玛丽的obj观测值（状态）为当前帧图片（像素），和人类玩超级玛丽一致，通过观察每一帧图像（大脑/模型）输出要执行的action

In [10]:
#创建env
from nes_py.wrappers import JoypadSpace
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

#借助包gym_super_mario_bros创建
env = gym_super_mario_bros.make('SuperMarioBros-v0')

#superMarioBros---v 其中：
#是{1，2，3，4，5，6，7，8}中的一个数字，表示世界 是{1，2，3，4}中的一个数字，
#表示一个世界中的阶段 是{0，1，2，3}中的一个数字，指定要使用的rom模式 0：标准ROM 1:降采样ROM 2：像素rom 3：矩形ROM

#初始化env
obj = env.reset()
print(obj.shape)

[[[104 136 252]
  [104 136 252]
  [104 136 252]
  ...
  [104 136 252]
  [104 136 252]
  [104 136 252]]

 [[104 136 252]
  [104 136 252]
  [104 136 252]
  ...
  [104 136 252]
  [104 136 252]
  [104 136 252]]

 [[104 136 252]
  [104 136 252]
  [104 136 252]
  ...
  [104 136 252]
  [104 136 252]
  [104 136 252]]

 ...

 [[240 208 176]
  [228  92  16]
  [228  92  16]
  ...
  [228  92  16]
  [228  92  16]
  [  0   0   0]]

 [[240 208 176]
  [228  92  16]
  [228  92  16]
  ...
  [228  92  16]
  [  0   0   0]
  [  0   0   0]]

 [[228  92  16]
  [  0   0   0]
  [  0   0   0]
  ...
  [  0   0   0]
  [  0   0   0]
  [228  92  16]]]
(240, 256, 3)


超级玛丽的观测值变成了一张240*256的rgb图片
为了验证，我们可视化出来

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(obj)

In [ ]:
#接下来看一下动作空间
print("env.action_space: ", env.action_space)

默认情况下， gym_super_mario_bros环境使用完整的NES操作空间256 离散动作。
为了解决这个问题，gym_super_mario_bros.actions提供 三个操作列表（RIGHT_ONLY、SIMPLE_MOVEMENT和COMPLEX_MOVEMENT） 
对于nes_py.wrappers.JoypadSpace包装器

In [ ]:
#我们选用SIMPLE_MOVEMENT来看下是否满足我们的通关需求
env = JoypadSpace(env, SIMPLE_MOVEMENT)
print("env.action_space: ", env.action_space)

7个基本动作包含了常用的操作 如上下左右，跳跃，右+跳，左+跳。
由此其实已经基本满足了常用的操作，而选择更多的动作反而会增加模型学习的难度。
所以我们选择SIMPLE_MOVEMENT模式即可

In [ ]:
#随机执行一个操作
obj, reward, done, info = env.step(1) #这里随机选择执行动作1
print("obj.shape", obj.shape)
print("reward", reward)
print("done", done)
print("info", info)

#3.3常用env Wrapper技巧

In [21]:
#先重新引入下相关包，防止报错
import gym_super_mario_bros
from gym.spaces import Box
from gym import Wrapper
from nes_py.wrappers import JoypadSpace#BinarySpaceToDiscreteSpaceEnv
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT, COMPLEX_MOVEMENT, RIGHT_ONLY
import cv2
import numpy as np
import subprocess as sp

## 3.3.1 rgb图像转灰度图
想象一下你在玩超级玛丽时如果把彩色图像换成灰度图，其实对你的操作并没有多大影响（只要能看出来障碍物即可判断路线和动作），
反而在模型训练中，rgb图像对算力和训练时间的要求会成倍增长，所以综合考虑咱们转换成灰度图才输入网络

In [1]:
#借助cv2即（opencv）包快速转换COLOR_RGB2GRAY
def process_frame(frame):
    if frame is not None:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY) #图像转换
        frame = cv2.resize(frame, (84, 84))[None, :, :] / 255. #裁剪合适大小，并归一化
        return frame
    else:
        return np.zeros((1, 84, 84))

## 3.3.2 SkipFrame
由于超级玛丽等游戏开发是面向玩家的（人），而非电脑，所以面向人类通关设计时，很多游戏帧是被放慢了，
比如执行一个action并不会立刻得到reard而是在接下来的几帧里才逐渐成效，换个通俗的说法，其实这么快速的游戏帧对我们并不需要，
我们只需要每秒能看到几帧就足以通关了，所以我们根据经验，每四帧只取一帧即可

In [2]:
class CustomSkipFrame(Wrapper):
    def __init__(self, env, skip=4):
        super(CustomSkipFrame, self).__init__(env)
        self.observation_space = Box(low=0, high=255, shape=(4, 84, 84))
        self.skip = skip

    def step(self, action):
        total_reward = 0
        states = []
        state, reward, done, info = self.env.step(action)
        for i in range(self.skip):
            if not done:
                state, reward, done, info = self.env.step(action)
                total_reward += reward
                states.append(state)
            else:
                states.append(state)
        states = np.concatenate(states, 0)[None, :, :, :]
        return states.astype(np.float32), reward, done, info

    def reset(self):
        state = self.env.reset()
        states = np.concatenate([state for _ in range(self.skip)], 0)[None, :, :, :]
        return states.astype(np.float32)

NameError: name 'Wrapper' is not defined

## 3.3.3 CustomReward
强化学习的优化目标必须是可量化的，所以在游戏里我们直接的优化目标就是最大化reward,
但是很多时候游戏直接设定的reward并不完全切合我们的实际目的（比如通关），或者在某个特定场景下（关卡下）不合适，
所以越是复杂的游戏场景，越是需要自定义reward来进行修正。
这里我们做了几个小优化如下：
1.reward += (info["score"] - self.curr_score) / 40.
原来的reward仅包含了对“离终点更近”的奖励和“时间消耗”、”死掉“的惩罚
为了让游戏更好玩，我们添加了info["score"]，包含了对获得技能、金币的奖励，但不是重点，为了不影响整体要通关的属性，弱化他
2.if done:
            if info["flag_get"]:
                reward += 50
            else:
                reward -= 50
我们对回合结束时到达终点和未达到的奖励和惩罚进行放大，激励agent更快速的到达终点
3.这里仅仅是对reward修改的一些示例，后面自己在实战时可以自己根据实际情况进行定义，比如当agent有时陷入一个错误的路线卡住时，可以添加一个缓冲区让agent学会后退等


In [24]:
class CustomReward(Wrapper):
    def __init__(self, env=None):
        super(CustomReward, self).__init__(env)
        self.observation_space = Box(low=0, high=255, shape=(1, 84, 84))
        self.curr_score = 0

    def step(self, action):
        state, reward, done, info = self.env.step(action)
        state = process_frame(state)
        reward += (info["score"] - self.curr_score) / 40.
        self.curr_score = info["score"]
        print(info)
        if done:
            if info["flag_get"]:
                reward += 50
            else:
                reward -= 50
        return state, reward / 10., done, info

    def reset(self):
        self.curr_score = 0
        return process_frame(self.env.reset())

In [25]:
#至此，我们完成了超级玛丽环境的自定义，封装如下：
def create_train_env(world, stage, action_type, output_path=None):
    env = gym_super_mario_bros.make("SuperMarioBros-{}-{}-v0".format(world, stage))
    if action_type == "right":
        actions = RIGHT_ONLY
    elif action_type == "simple":
        actions = SIMPLE_MOVEMENT
    else:
        actions = COMPLEX_MOVEMENT
    env = JoypadSpace(env, actions)
    env = CustomReward(env)
    env = CustomSkipFrame(env)
    return env, env.observation_space.shape[0], len(actions)

In [26]:
#测试一下
custom_env = create_train_env(1,1,'simple')
print(custom_env)


(<CustomSkipFrame<CustomReward<JoypadSpace<TimeLimit<SuperMarioBrosEnv<SuperMarioBros-1-1-v0>>>>>>, 4, 7)


# R4 PPO（近段策略优化）算法讲解

1、策略（要输出最优动作的策略模型）

2、近端（代理函数的剪裁）

3、优化（使用代理函数）的出现及其实际意义，导致了算法的命名。

## R4.1 由浅入深，简化版ppo

In [3]:
#导入gym和torch相关包
import gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

In [4]:
#Hyperparameters
learning_rate = 0.0005 #学习率
gamma         = 0.98   #
lmbda         = 0.95
eps_clip      = 0.1
K_epoch       = 3
T_horizon     = 20

In [5]:
#定义PPO架构
class PPO(nn.Module):
    def __init__(self):
        super(PPO, self).__init__()
        self.data = [] #用来存储交互数据
        
        self.fc1   = nn.Linear(4,256) #由于倒立摆环境简单，这里仅用一个线性变换来训练数据
        self.fc_pi = nn.Linear(256,2) #policy函数（输出action）的全连接层
        self.fc_v  = nn.Linear(256,1) #value函数（输出v）的全连接层
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate) #优化器

    #policy函数
    #输入观测值x
    #输出动作空间概率，从而选择最优action
    def pi(self, x, softmax_dim = 0): 
        x = F.relu(self.fc1(x))
        x = self.fc_pi(x)
        prob = F.softmax(x, dim=softmax_dim)
        return prob
    
    #value函数
    #输入观测值x
    #输出x状态下value的预测值（reward）,提供给policy函数作为参考值
    def v(self, x):
        x = F.relu(self.fc1(x))
        v = self.fc_v(x)
        return v
    
    #把交互数据存入buffer
    def put_data(self, transition):
        self.data.append(transition)
        
    #把数据形成batch，训练模型时需要一个一个batch输入模型
    def make_batch(self):
        s_lst, a_lst, r_lst, s_prime_lst, prob_a_lst, done_lst = [], [], [], [], [], []
        for transition in self.data:
            s, a, r, s_prime, prob_a, done = transition
            
            s_lst.append(s)
            a_lst.append([a])
            r_lst.append([r])
            s_prime_lst.append(s_prime)
            prob_a_lst.append([prob_a])
            done_mask = 0 if done else 1
            done_lst.append([done_mask])
            
        s,a,r,s_prime,done_mask, prob_a = torch.tensor(s_lst, dtype=torch.float), torch.tensor(a_lst), \
                                          torch.tensor(r_lst), torch.tensor(s_prime_lst, dtype=torch.float), \
                                          torch.tensor(done_lst, dtype=torch.float), torch.tensor(prob_a_lst)
        self.data = []
        return s, a, r, s_prime, done_mask, prob_a
    
    
    #训练模型
    
    def train_net(self):
        #make batch 数据，喂给模型
        s, a, r, s_prime, done_mask, prob_a = self.make_batch()

        for i in range(K_epoch): #K_epoch：训练多少个epoch
            #计算td_error 误差，value模型的优化目标就是尽量减少td_error
            td_target = r + gamma * self.v(s_prime) * done_mask
            delta = td_target - self.v(s)
            delta = delta.detach().numpy()

            #计算advantage:
            #即当前策略比一般策略（baseline）要好多少
            #policy的优化目标就是让当前策略比baseline尽量好，但是每次更新时又不能偏离太多，所以后面会有个clip
            advantage_lst = []
            advantage = 0.0
            for delta_t in delta[::-1]:
                advantage = gamma * lmbda * advantage + delta_t[0]
                advantage_lst.append([advantage])
            advantage_lst.reverse()
            advantage = torch.tensor(advantage_lst, dtype=torch.float)

            #计算ratio 防止单词更新偏离太多
            pi = self.pi(s, softmax_dim=1)
            pi_a = pi.gather(1,a)
            ratio = torch.exp(torch.log(pi_a) - torch.log(prob_a))  # a/b == exp(log(a)-log(b))

            #通过clip 保证ratio在（1-eps_clip, 1+eps_clip）范围内
            surr1 = ratio * advantage
            surr2 = torch.clamp(ratio, 1-eps_clip, 1+eps_clip) * advantage
            #这里简化ppo，把policy loss和value loss放在一起计算
            loss = -torch.min(surr1, surr2) + F.smooth_l1_loss(self.v(s) , td_target.detach())

            #梯度优化
            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()

In [ ]:
#主函数：简化ppo 这里先交互T_horizon个回合然后停下来学习训练，再交互，这样循环10000次
def main():
    #创建倒立摆环境
    env = gym.make('CartPole-v1')
    model = PPO()
    score = 0.0
    print_interval = 20

    #主循环
    for n_epi in range(100):
        s = env.reset()
        done = False
        while not done:
            for t in range(T_horizon):
                #由当前policy模型输出最优action
                prob = model.pi(torch.from_numpy(s).float())
                m = Categorical(prob)
                a = m.sample().item()
                #用最优action进行交互
                s_prime, r, done, info = env.step(a)
                #存储交互数据，等待训练
                model.put_data((s, a, r/100.0, s_prime, prob[a].item(), done))
                s = s_prime

                score += r
                if done:
                    break

            #模型训练
            model.train_net()

        #打印每轮的学习成绩
        if n_epi%print_interval==0 and n_epi!=0:
            print("# of episode :{}, avg score : {:.1f}".format(n_epi, score/print_interval))
            score = 0.0

    env.close()

if __name__ == '__main__':
    main()